# Seeing Data: instructor solution

This executable version supplies the technical solution and suggested interpretation. Use it for demonstration and troubleshooting, not as a replacement for student reasoning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 3)


In [ ]:
x_common = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
x_four = [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8]
values = {
    'A': (x_common, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    'B': (x_common, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    'C': (x_common, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    'D': (x_four,   [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
anscombe = pd.concat(
    [pd.DataFrame({'dataset': label, 'x': x, 'y': y}) for label, (x, y) in values.items()],
    ignore_index=True,
)
anscombe.head()


## Shared summaries

In [ ]:
summary_rows = []
for label, group in anscombe.groupby('dataset'):
    slope, intercept = np.polyfit(group['x'], group['y'], 1)
    summary_rows.append({
        'dataset': label,
        'x_mean': group['x'].mean(),
        'y_mean': group['y'].mean(),
        'x_variance': group['x'].var(ddof=1),
        'y_variance': group['y'].var(ddof=1),
        'correlation': group['x'].corr(group['y']),
        'slope': slope,
        'intercept': intercept,
    })
summary = pd.DataFrame(summary_rows)
summary


In [ ]:
assert summary.shape == (4, 8)
assert set(summary['dataset']) == {'A', 'B', 'C', 'D'}
assert np.allclose(summary['x_mean'], 9.0, atol=0.01)
assert np.allclose(summary['y_mean'], 7.5, atol=0.01)
assert np.allclose(summary['correlation'], 0.816, atol=0.01)
print('Summary checks passed.')


## Common-scale small multiples

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8), sharex=True, sharey=True)
for ax, (label, group) in zip(axes.flat, anscombe.groupby('dataset')):
    slope, intercept = np.polyfit(group['x'], group['y'], 1)
    line_x = np.array([3, 20])
    ax.scatter(group['x'], group['y'], s=55, color='#18678f')
    ax.plot(line_x, slope * line_x + intercept, color='#e66852', linewidth=2)
    ax.set_title(f'Dataset {label}', fontweight='bold')
    ax.set_xlim(3, 20)
    ax.set_ylim(2, 14)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
fig.suptitle('Similar summaries, different structures', fontsize=16, fontweight='bold')
fig.tight_layout()
plt.show()


## Generate and verify

Real work now: the `transport` table below is bigger than anything you want to
read by eye. **Predict first** (one sentence: what should total riders per mode
look like, 2019–2025?), then give your assistant a bounded task, draft
`plot_mode_recovery` from its docstring, and then run the verification cell.
Nothing counts until the checks pass.

In [ ]:
# The transport evidence, run this cell, no need to edit it.
# Synthetic monthly rider counts for six NSW regions and four transport modes,
# 2019-2025, with a seasonal cycle and a COVID-shaped shock. Teaching data.
import math, random
random.seed(42)

REGIONS = ["Inner Sydney", "Western Sydney", "Northern Beaches",
           "Central Coast", "Newcastle", "Illawarra"]
MODES = ["Train", "Bus", "Ferry", "Light rail"]
BASE = {"Train": 1_000_000, "Bus": 700_000, "Ferry": 90_000, "Light rail": 120_000}
FACTOR = {"Inner Sydney": 1.3, "Western Sydney": 1.1, "Northern Beaches": 0.55,
          "Central Coast": 0.45, "Newcastle": 0.5, "Illawarra": 0.42}

rows = []
for region in REGIONS:
    for mode in MODES:
        base = BASE[mode] * FACTOR[region]
        for date in pd.date_range("2019-01-01", "2025-12-01", freq="MS"):
            season = 1 + 0.08 * math.sin((date.month - 1) / 12 * 2 * math.pi)
            covid = 1.0
            if pd.Timestamp("2020-03-01") <= date <= pd.Timestamp("2021-12-01"):
                covid = 0.35 + 0.3 * (date - pd.Timestamp("2020-03-01")).days / 640
            elif date > pd.Timestamp("2021-12-01"):
                covid = min(1.0, 0.65 + 0.35 * (date - pd.Timestamp("2021-12-01")).days / 1100)
            rows.append({"date": date, "region": region, "mode": mode,
                         "riders": int(base * season * covid * random.gauss(1, 0.03))})

transport = pd.DataFrame(rows)
print(f"{len(transport):,} rows")
transport.head()

In [ ]:
def plot_mode_recovery(transport: pd.DataFrame) -> pd.DataFrame:
    agg = transport.groupby(["date", "mode"], as_index=False)["riders"].sum()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for mode, g in agg.groupby("mode"):
        ax.plot(g["date"], g["riders"] / 1e6, label=mode)
    ax.set_ylabel("riders (millions)")
    ax.set_title("Patronage collapsed in 2020-21 and has largely recovered")
    ax.legend()
    plt.show()
    return agg


plotted = plot_mode_recovery(transport)
plotted.head()

In [ ]:
# Verification: the same ladder you will climb for every generated chart.
# TRACE: the plotted table must reconcile with the source table.
assert plotted["riders"].sum() == transport["riders"].sum(), (
    "Aggregation lost or duplicated riders: plotted total != source total")
# CHECK: every mode present, every month present, no NaNs.
assert set(plotted["mode"]) == set(MODES), "A mode went missing in the aggregation"
assert plotted.groupby("mode")["date"].nunique().eq(84).all(), (
    "Each mode should have 84 monthly points (2019-01..2025-12)")
assert plotted["riders"].notna().all(), "NaNs appeared during aggregation"
# Spot total, hand-derivable: Train riders in Jan 2019 across all six regions.
jan = pd.Timestamp("2019-01-01")
jan_train = plotted[(plotted["mode"] == "Train") & (plotted["date"] == jan)]["riders"].iloc[0]
source_jan_train = transport[(transport["mode"] == "Train") & (transport["date"] == jan)]["riders"].sum()
assert jan_train == source_jan_train, "Spot total disagrees for Train, Jan 2019"
print("TRACE ✓  CHECK ✓, now TEST: does the recovery story survive a per-region view?")

## Suggested interpretation

- A resembles a conventional linear relationship.
- B is curved, so the fitted line misses systematic structure.
- C is strongly influenced by one vertical outlier.
- D is strongly influenced by one high-leverage horizontal point.
- Similar summary statistics do not establish similar data-generating structure.

Suggested repair of generated prose:

> The datasets share similar summary statistics and fitted lines, but their plots reveal different structures, including curvature and influential outliers; the summaries alone do not justify a common model.

## Deliberately misleading view and repair

In [ ]:
dataset_c = anscombe.query("dataset == 'C'")
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4.2))
left.scatter(dataset_c['x'], dataset_c['y'], color='#e66852', s=55)
left.set(xlim=(3, 20), ylim=(2, 14), title='Misleading crop: outlier removed from view', xlabel='x', ylabel='y')
left.set_ylim(4, 10)

right.scatter(dataset_c['x'], dataset_c['y'], color='#18678f', s=55)
right.set(xlim=(3, 20), ylim=(2, 14), title='Repair: full common scale', xlabel='x', ylabel='y')
fig.tight_layout()
plt.show()


## Provenance model

Record the tool and model, prompt purpose, output used, modifications, verification and what the tool missed. A complete record explains decisions rather than pasting a transcript.